# 04 — Readmission Analysis

Detects 30-day hospital readmissions and identifies which diagnoses, DRGs, and beneficiary groups have elevated readmission risk.

A readmission occurs when a patient is discharged and readmitted within 30 days.

Requires `cms_data.db` from `01_setup.ipynb`.

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

In [2]:
DB = Path("../cms_data.db")
conn = sqlite3.connect(DB)

---
## 1. Flag 30-day readmissions

In [3]:
# Load inpatient claims, sort by patient and date
claims = pd.read_sql("""
SELECT
  DESYNPUF_ID,
  CLM_ID,
  CLM_ADMSN_DT,
  NCH_BENE_DSCHRG_DT,
  CLM_DRG_CD,
  ICD9_DGNS_CD_1,
  CLM_PMT_AMT
FROM inpatient_claims
WHERE CLM_ADMSN_DT IS NOT NULL
ORDER BY DESYNPUF_ID, CLM_ADMSN_DT
""", conn)

# Convert date strings to datetime
claims['CLM_ADMSN_DT'] = pd.to_datetime(claims['CLM_ADMSN_DT'], format='%Y%m%d')
claims['NCH_BENE_DSCHRG_DT'] = pd.to_datetime(claims['NCH_BENE_DSCHRG_DT'], format='%Y%m%d')

# For each patient, calculate days to next admission
claims['next_admission_date'] = claims.groupby('DESYNPUF_ID')['CLM_ADMSN_DT'].shift(-1)
claims['days_to_readmission'] = (claims['next_admission_date'] - claims['NCH_BENE_DSCHRG_DT']).dt.days

# Flag readmission if next admission within 30 days
claims['readmitted_30d'] = (claims['days_to_readmission'] > 0) & (claims['days_to_readmission'] <= 30)

print(f"Total inpatient stays: {len(claims):,}")
print(f"Readmitted within 30 days: {claims['readmitted_30d'].sum():,} ({claims['readmitted_30d'].mean()*100:.1f}%)")
print(f"Not readmitted (or >30 days): {(~claims['readmitted_30d']).sum():,}")

Total inpatient stays: 66,773
Readmitted within 30 days: 6,423 (9.6%)
Not readmitted (or >30 days): 60,350


---
## 2. Readmission rates by DRG

In [4]:
drg_readmit = claims.groupby('CLM_DRG_CD').agg(
    num_admissions=('CLM_ID', 'count'),
    readmissions=('readmitted_30d', 'sum')
).reset_index()

drg_readmit['readmission_rate'] = (drg_readmit['readmissions'] / drg_readmit['num_admissions'] * 100).round(1)
drg_readmit = drg_readmit.sort_values('readmission_rate', ascending=False).reset_index(drop=True)

print("DRGs with Highest 30-Day Readmission Rates (top 20)")
print("="*80)
display(drg_readmit.head(20))
drg_readmit.to_csv("../data/exports/readmission_rates_by_drg.csv", index=False)

DRGs with Highest 30-Day Readmission Rates (top 20)


,CLM_DRG_CD,num_admissions,readmissions,readmission_rate
0,735,14,5,35.7
1,767,3,1,33.3
2,959,10,3,30.0
3,963,12,3,25.0
4,714,12,3,25.0
5,928,4,1,25.0
6,956,13,3,23.1
7,813,58,13,22.4
8,150,18,4,22.2
9,987,28,6,21.4


---
## 3. Readmission rates by primary diagnosis

In [5]:
diag_readmit = claims[claims['ICD9_DGNS_CD_1'].notna()].groupby('ICD9_DGNS_CD_1').agg(
    num_admissions=('CLM_ID', 'count'),
    readmissions=('readmitted_30d', 'sum')
).reset_index()

diag_readmit['readmission_rate'] = (diag_readmit['readmissions'] / diag_readmit['num_admissions'] * 100).round(1)
diag_readmit = diag_readmit.sort_values('readmission_rate', ascending=False).reset_index(drop=True)

print("Primary Diagnoses with Highest 30-Day Readmission Rates (top 20)")
print("="*80)
display(diag_readmit.head(20))
diag_readmit.to_csv("../data/exports/readmission_rates_by_diagnosis.csv", index=False)

Primary Diagnoses with Highest 30-Day Readmission Rates (top 20)


,ICD9_DGNS_CD_1,num_admissions,readmissions,readmission_rate
0,0071,1,1,100.0
1,3559,1,1,100.0
2,3568,2,2,100.0
3,V556,1,1,100.0
4,34989,1,1,100.0
5,34670,1,1,100.0
6,01500,1,1,100.0
7,01300,1,1,100.0
8,3335,1,1,100.0
9,3203,1,1,100.0


---
## 4. Frequent readmitters (beneficiaries with multiple readmissions)

In [6]:
# Count readmissions per beneficiary
readmitter_profile = claims.groupby('DESYNPUF_ID').agg(
    total_admits=('CLM_ID', 'count'),
    num_readmitted=('readmitted_30d', 'sum'),
    avg_cost_per_admit=('CLM_PMT_AMT', 'mean')
).reset_index()

readmitter_profile['readmit_rate'] = (readmitter_profile['num_readmitted'] / readmitter_profile['total_admits'] * 100).round(1)

# Identify frequent readmitters (2+ readmissions)
frequent_readmitters = readmitter_profile[readmitter_profile['num_readmitted'] >= 2].sort_values('num_readmitted', ascending=False)

print(f"\nBeneficiaries with ≥2 readmissions: {len(frequent_readmitters):,}")
print(f"Total readmissions from this group: {frequent_readmitters['num_readmitted'].sum():,}")
print(f"\nAvg characteristics:")
print(f"  Total admits: {frequent_readmitters['total_admits'].mean():.1f}")
print(f"  Readmission rate: {frequent_readmitters['readmit_rate'].mean():.1f}%")
print(f"  Avg cost per admit: ${frequent_readmitters['avg_cost_per_admit'].mean():,.0f}")

print(f"\nTop 20 frequent readmitters:")
display(frequent_readmitters.head(20))


Beneficiaries with ≥2 readmissions: 1,016
Total readmissions from this group: 2,359

Avg characteristics:
  Total admits: 5.3
  Readmission rate: 47.4%
  Avg cost per admit: $10,017

Top 20 frequent readmitters:


,DESYNPUF_ID,total_admits,num_readmitted,avg_cost_per_admit,readmit_rate
35000,ED29D09DDA6FA023,8,6,15875.000000,75.0
30325,CD57573A5B77AAFE,14,6,14571.428571,42.9
30521,CE7D6042D4536BA6,11,6,9272.727273,54.5
27601,BB0BE288F5F8DE25,9,6,11444.444444,66.7
19227,8122C692D8B798E8,8,6,11000.000000,75.0
22348,96D9E8DE1DFFE60A,9,6,11555.555556,66.7
21988,943160E50517FA93,8,5,11000.000000,62.5
25214,AAF90A580D7C5362,9,5,11777.777778,55.6
5104,21CC2F66DCC830BA,10,5,8600.000000,50.0
13787,5C557CCF0C623AD5,9,5,9888.888889,55.6


---
## 5. Readmission risk by chronic condition status

In [7]:
# Join claims with beneficiary chronic condition data
bene_chronic = pd.read_sql("""
SELECT DISTINCT
  DESYNPUF_ID,
  SP_CHF,
  SP_COPD,
  SP_DIABETES,
  SP_ISCHMCHT,
  SP_DEPRESSN,
  SP_CHRNKIDN
FROM beneficiary_summary
""", conn)

claims_with_chronic = claims.merge(bene_chronic, on='DESYNPUF_ID', how='left')

chronic_readmit = pd.DataFrame()
for condition in ['SP_CHF', 'SP_COPD', 'SP_DIABETES', 'SP_ISCHMCHT', 'SP_DEPRESSN', 'SP_CHRNKIDN']:
    with_cond = claims_with_chronic[claims_with_chronic[condition] == 1]
    without_cond = claims_with_chronic[claims_with_chronic[condition] == 2]
    
    row = {
        'condition': condition,
        'readmit_rate_with_condition': (with_cond['readmitted_30d'].sum() / len(with_cond) * 100).round(1) if len(with_cond) > 0 else None,
        'readmit_rate_without_condition': (without_cond['readmitted_30d'].sum() / len(without_cond) * 100).round(1) if len(without_cond) > 0 else None,
    }
    chronic_readmit = pd.concat([chronic_readmit, pd.DataFrame([row])], ignore_index=True)

chronic_readmit['risk_increase'] = (chronic_readmit['readmit_rate_with_condition'] - chronic_readmit['readmit_rate_without_condition']).round(1)

print("30-Day Readmission Rates: With vs Without Chronic Conditions")
print("="*90)
display(chronic_readmit)
chronic_readmit.to_csv("../data/exports/readmission_by_chronic_condition.csv", index=False)

30-Day Readmission Rates: With vs Without Chronic Conditions


,condition,readmit_rate_with_condition,readmit_rate_without_condition,risk_increase
0,SP_CHF,11.0,7.0,4.0
1,SP_COPD,12.0,8.1,3.9
2,SP_DIABETES,10.4,7.7,2.7
3,SP_ISCHMCHT,10.3,7.2,3.1
4,SP_DEPRESSN,10.9,8.5,2.4
5,SP_CHRNKIDN,11.8,7.6,4.2


In [8]:
conn.close()
print("\n✓ Readmission analysis complete. Outputs saved to data/exports/")


✓ Readmission analysis complete. Outputs saved to data/exports/
